# Rol 1 — El Codificador: Representación del Cromosoma y Extracción de Melodía

**Curso:** Inteligencia Artificial — Examen Práctico Grupal: *Del Silbido al Instrumento*
**Instrumento Objetivo:** Flauta traversa / dulce
**Responsable:** Rol 1 (El Codificador)

---

## Resumen del Rol
El Rol 1 es responsable de la capa fundamental del Algoritmo Genético:
1. **Extracción y procesamiento acústico:** Detección de frecuencia fundamental ($), identificación de notas MIDI y duraciones temporales desde el silbido.
2. **Representación genotípica:** Definición del cromosoma en espacio continuo real 1^D$, límites físicos, funciones de codificación/decodificación e invertibilidad.
3. **Análisis del espacio de búsqueda:** Cuantificación formal de la combinatoria y justificación de por qué la metaheurística del AG es necesaria frente a la búsqueda exhaustiva.
4. **Desacoplamiento arquitectónico:** Esquema extensible () para permitir ajustes o nuevas variables acústicas sin romper los módulos de los demás roles.

In [ ]:
# 1. Configuración del entorno e importaciones
import sys
from pathlib import Path

# Asegurar que el directorio raíz del proyecto esté en el PATH
PROYECTO_RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROYECTO_RAIZ) not in sys.path:
    sys.path.insert(0, str(PROYECTO_RAIZ))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import src.representacion as rep
import src.extraccion as ext

print("Módulos del Rol 1 cargados exitosamente.")
print(f"Número de genes en el cromosoma canónico: {rep.num_genes()}")

---
## 2. Detección de Pitch y Extracción de la Melodía

Para asegurar la independencia del pipeline mientras se captura la grabación humana final, el módulo  cuenta con un generador sintético calibrado con propiedades realistas (vibrato de ~5.5 Hz, turbulencia de aire y transitorios suaves de embocadura).

In [ ]:
# Generar o cargar el silbido de trabajo
ruta_silbido = PROYECTO_RAIZ / "datos" / "audio" / "silbidos" / "silbido_01.wav"

if not ruta_silbido.is_file():
    print("Generando silbido mock sintético para validación...")
    audio_silbido = ext.generar_silbido_sintetico()
    ext.guardar_audio(ruta_silbido, audio_silbido)
else:
    print(f"Cargando silbido existente desde: {ruta_silbido}")
    audio_silbido, _ = ext.cargar_audio(ruta_silbido)

# Diagnóstico de calidad acústica
diag = ext.diagnosticar_audio_silbido(audio_silbido)
print("\n--- DIAGNÓSTICO ACÚSTICO ---")
for k, v in diag.items():
    print(f"{k}: {v}")

In [ ]:
# Detección de F0 tramo a tramo mediante pYIN / autocorrelación
f0, tiempos, sonoridad = ext.detectar_f0(audio_silbido)

# Segmentación en notas MIDI y duraciones
notas_midi, duraciones_s = ext.segmentar_melodia(f0, tiempos, sonoridad)

# Guardar y verificar CSV
ruta_csv = PROYECTO_RAIZ / "datos" / "procesados" / "melodia_01.csv"
ext.guardar_melodia_csv(notas_midi, duraciones_s, ruta_csv)

# Visualización en tabla pandas
df_melodia = pd.DataFrame({
    "Nota MIDI": notas_midi,
    "Nombre Nota": [ext.nombre_nota_midi(n) for n in notas_midi],
    "Frecuencia Hz": [round(float(ext.midi_a_hz(n)), 2) if n > 0 else 0.0 for n in notas_midi],
    "Duración (s)": duraciones_s,
})
display(df_melodia)

In [ ]:
# Gráfica de la forma de onda con el contorno de F0 detectado
plt.figure(figsize=(12, 5))
t_audio = np.linspace(0, len(audio_silbido) / 22050, len(audio_silbido))

plt.subplot(2, 1, 1)
plt.plot(t_audio, audio_silbido, color="steelblue", alpha=0.8)
plt.title("Forma de onda del silbido grabado (Normalizado WAV mono 22050 Hz)")
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud")
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
f0_plot = np.where(f0 > 0, f0, np.nan)
plt.plot(tiempos, f0_plot, color="darkred", lw=2, label="F0 detectada (Hz)")
# Graficar líneas guía con las notas principales extraídas
notas_unicas = [n for n in np.unique(notas_midi) if n > 0][:4]
colores = ["gray", "darkgreen", "purple", "orange"]
for n, c in zip(notas_unicas, colores):
    plt.axhline(ext.midi_a_hz(n), color=c, linestyle="--", alpha=0.7, label=ext.nombre_nota_midi(n))
plt.title("Contorno de F0 (Pitch Tracking sobre silbido real)")
plt.xlabel("Tiempo (s)")
plt.ylabel("Frecuencia (Hz)")
plt.legend(loc="upper right", fontsize=9)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 3. Especificación Formal de los Genes y Contrato Acústico

El genotipo consiste en un vector normalizado 1^8$ que mapea linealmente a los parámetros fenotípicos de la síntesis de la flauta.

In [ ]:
# Construcción de la tabla de genes formal
tabla_genes = []
for i, g in enumerate(rep.GENES_SCHEMA):
    tabla_genes.append({
        "Índice": i,
        "Gen": g.nombre,
        "Unidad": g.unidad,
        "Mínimo": g.min_val,
        "Máximo": g.max_val,
        "Default Flauta": g.default,
        "Justificación Física / Acústica": g.descripcion
    })

df_genes = pd.DataFrame(tabla_genes)
display(df_genes)

---
## 4. Pruebas Unitarias de Codificación e Invertibilidad

La rúbrica exige que la codificación sea bidireccional y robusta: $	ext{decodificar}(	ext{codificar}(x)) pprox x$.

In [ ]:
# Test de invertibilidad con valores por defecto
test_ok = rep.probar_inversion_codificacion()
print(f"¿Test formal decodificar(codificar(x)) == x exitoso?: {test_ok}")
assert test_ok, "Fallo en la prueba de invertibilidad básica"

# Test con 1000 individuos aleatorios en el hipercubo
rng = np.random.default_rng(2024)
errores_max = []
for _ in range(1000):
    ind_aleatorio = rep.generar_individuo_aleatorio(normalizado=True, rng=rng)
    fenotipo = rep.decodificar(ind_aleatorio)
    recodificado = rep.codificar(fenotipo)
    err = np.max(np.abs(ind_aleatorio - recodificado))
    errores_max.append(err)

print(f"Error máximo de reconstrucción en 1,000 muestras aleatorias: {max(errores_max):.2e}")
assert max(errores_max) < 1e-6, "Error de precisión flotante fuera de tolerancia tolerada"

---
## 5. Análisis Matemático del Espacio de Búsqueda

Demostración rigurosa del tamaño del espacio de búsqueda para la justificación teórica requerida en el informe final.

**Criterio de discretización:** Discretización de paso absoluto 0.001 en las unidades físicas de cada gen; la cantidad de niveles depende de su rango e incluye los extremos.

Cada gen $i$ con rango $[\min_i, \max_i]$ y resolución $\Delta = 0.001$ aporta $\lfloor(\max_i - \min_i)/\Delta\rfloor + 1$ niveles.


In [ ]:
espacio_analisis = rep.calcular_tamano_espacio_busqueda(resolucion=1e-3)

print("=== ANÁLISIS DEL ESPACIO DE BÚSQUEDA ===")
print(f"Número de genes (dimensiones): {espacio_analisis['num_genes']}")
print(f"Resolución de discretización considerada: {espacio_analisis['resolucion_utilizada']}")
print(f"Estados posibles por gen: {espacio_analisis['estados_por_gen']}")
print(f"Espacio de búsqueda total: {espacio_analisis['estados_totales_notacion_cientifica']} combinaciones")
print(f"Equivalente en bits de información: {espacio_analisis['bits_equivalentes']:.2f} bits")
print(f"Tiempo estimado por fuerza bruta (@1,000 evals/s): {espacio_analisis['tiempo_estimado_anos']:.2e} años")

---
## 6. Análisis de Sensibilidad Preliminar: Variación de Genes

Inspección del efecto en el fenotipo al perturbar genes clave de envolvente y armónicos.

In [ ]:
# Simulación de envolventes ADSR variando el tiempo de ataque
dur_nota = 0.60
sr_demo = 22050
n_samples = int(dur_nota * sr_demo)
t_env = np.linspace(0, dur_nota, n_samples)

ataques_prueba = [0.01, 0.05, 0.12, 0.22]
plt.figure(figsize=(10, 4))

for att in ataques_prueba:
    n_att = int(att * sr_demo)
    n_dec = int(0.08 * sr_demo)
    n_rel = int(0.06 * sr_demo)
    n_sost = max(0, n_samples - n_att - n_dec - n_rel)
    
    env = np.concatenate([
        np.linspace(0, 1.0, n_att),
        np.linspace(1.0, 0.8, n_dec),
        np.full(n_sost, 0.8),
        np.linspace(0.8, 0, n_rel)
    ])[:n_samples]
    plt.plot(t_env, env, lw=2, label=f"Ataque = {att*1000:.0f} ms")

plt.title("Impacto fenotípico de la variación del gen 'ataque_s'")
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud de la envolvente")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. Pregunta Teórica del Rol 1 (Para el Informe Final)

> **Pregunta Oficial:** *¿Cómo influye la representación del cromosoma en la eficiencia del AG? Da un ejemplo del proyecto.*

### Respuesta Técnica Estructurada:

1. **Continuidad vs. Acantilados de Hamming (Hamming Cliffs):**
   Si hubiéramos optado por una representación binaria clásica para los parámetros acústicos (por ejemplo, cuantizando el tiempo de ataque en 8 bits), dos valores fenotípicamente casi idénticos como 27$ () y 28$ () distan a 8 mutaciones de bits consecutivas. Esto genera discontinuidades y valles artificiales en la superficie de aptitud. Al utilizar una **representación continua real en $\mathbb{R}^8*, garantizamos que pequeñas perturbaciones en el genotipo correspondan a pequeñas variaciones acústicas en el fenotipo, preservando la localidad del espacio.

2. **Epistasis e Interdependencia Génica en el Proyecto:**
   En la síntesis de flauta, los genes no actúan de manera aislada. Los armónicos (, a_2, a_3$) interactúan multiplicativamente con la envolvente ADSR (, D, S, R$) y con el nivel de soplo ($). Si el nivel de sostenido $ es nulo o el ataque excede la duración total de la nota, la modulación armónica se vuelve inaudible (fenómeno de neutralidad o epistasis fuerte). Por ello, el cromosoma desacoplado y normalizado con validación de envolventes evita regiones muertas en el espacio de búsqueda.

3. **Conclusión:**
   La elección de una representación en vector real acotado con operadores genéticos compatibles (como cruce Blend-$lpha$ y mutación gaussiana con clipping) maximiza la suavidad de la función objetivo, permitiendo que la población converja eficientemente hacia el timbre de flauta sin quedar atrapada en mínimos locales arbitrarios creados por una mala codificación.